# Titanic Survival Prediction — Machine Learning

**Kaggle competition:** Titanic - Machine Learning from Disaster  
**Goal:** Predict whether a passenger survived the Titanic disaster.

### Project highlights
- End-to-end binary classification workflow
- Data inspection and missing-value analysis
- Feature engineering from passenger names, families, cabins and tickets
- Baseline Logistic Regression model
- Improved CatBoost ensemble with stratified cross-validation
- Kaggle submission generation

**Best public leaderboard score achieved during experimentation: 0.80622**

> The notebook is intentionally organized as a clean, reproducible project rather than a record of every experiment.

## 1. Imports and reproducibility

We import the libraries used for data manipulation, validation and modeling.  
A fixed random seed makes the evaluation easier to reproduce.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from catboost import CatBoostClassifier

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

## 2. Load the Titanic data

`train.csv` contains the target column `Survived`.  
`test.csv` contains the passengers for whom Kaggle expects predictions.

In [ ]:
DATA_CANDIDATES = [
    Path("/kaggle/input/competitions/titanic"),
    Path("/kaggle/input/titanic"),
    Path("."),
]

DATA_DIR = next(
    (path for path in DATA_CANDIDATES
     if (path / "train.csv").exists() and (path / "test.csv").exists()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Titanic train.csv and test.csv were not found. "
        "On Kaggle, attach the Titanic competition data first."
    )

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)
train.head()

## 3. Quick data inspection

Before modeling, we check:
- dataset size,
- missing values,
- the proportion of survivors.

This helps us decide which preprocessing steps are necessary.

In [ ]:
missing_values = pd.DataFrame({
    "Missing": train.isna().sum(),
    "Missing_%": (train.isna().mean() * 100).round(1)
}).sort_values("Missing", ascending=False)

print("Survival rate:", round(train["Survived"].mean(), 3))
missing_values

## 4. Baseline model — Logistic Regression

We first build a simple and interpretable baseline using:
`Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, and `Embarked`.

Numerical missing values are filled with the median.  
Categorical missing values are filled with the most frequent value and one-hot encoded.

The model is evaluated with **5-fold stratified cross-validation**, which is more reliable than relying on one train/test split.

In [ ]:
baseline_features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"
]

X_baseline = train[baseline_features]
y = train["Survived"]

numerical_baseline = ["Age", "SibSp", "Parch", "Fare"]
categorical_baseline = ["Pclass", "Sex", "Embarked"]

baseline_preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numerical_baseline
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_baseline
    )
])

baseline_model = Pipeline([
    ("preprocessing", baseline_preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

baseline_scores = cross_val_score(
    baseline_model,
    X_baseline,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Baseline fold scores:", baseline_scores.round(4))
print("Baseline mean accuracy:", round(baseline_scores.mean(), 4))

## 5. Feature engineering

The improved model uses information that is present in the raw dataset but is not directly usable as a simple number.

Created features:
- **Title**: extracted from the passenger name (`Mr`, `Mrs`, `Miss`, `Master`, etc.)
- **FamilySize**: number of relatives traveling together + the passenger
- **IsAlone**: whether the passenger traveled without close family
- **Deck / HasCabin**: information derived from the cabin field
- **AgeMissing**: whether age was originally missing
- **Ticket / TicketPrefix**: ticket information
- **FamilyKey**: surname combined with family size
- **SexClass**: interaction between sex and passenger class

These features are useful because survival was strongly related to passenger demographics, class, and traveling groups.

In [ ]:
CAT_COLUMNS = [
    "Pclass", "Sex", "Embarked", "Title", "Deck",
    "Ticket", "TicketPrefix", "FamilyKey", "SexClass"
]

def build_features(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()
    out = pd.DataFrame(index=df.index)

    # Basic variables
    out["Pclass"] = df["Pclass"].astype(str)
    out["Sex"] = df["Sex"].fillna("Unknown").astype(str)
    out["Embarked"] = df["Embarked"].fillna("Unknown").astype(str)
    out["Age"] = df["Age"]
    out["Fare"] = df["Fare"]
    out["SibSp"] = df["SibSp"]
    out["Parch"] = df["Parch"]

    # Title extracted from the passenger name
    title = (
        df["Name"]
        .str.extract(r",\s*([^.]*)\.", expand=False)
        .str.strip()
        .replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    )
    main_titles = {"Mr", "Miss", "Mrs", "Master"}
    out["Title"] = title.where(title.isin(main_titles), "Rare")

    # Family-related features
    out["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)

    surname = (
        df["Name"]
        .str.split(",")
        .str[0]
        .str.strip()
        .str.upper()
    )
    out["FamilyKey"] = surname + "_" + out["FamilySize"].astype(str)
    out.loc[out["FamilySize"] == 1, "FamilyKey"] = "SOLO"

    # Cabin-related features
    out["AgeMissing"] = df["Age"].isna().astype(int)
    out["HasCabin"] = df["Cabin"].notna().astype(int)
    out["Deck"] = df["Cabin"].str[0].fillna("Unknown").astype(str)

    # Ticket-related features
    ticket = df["Ticket"].fillna("Unknown").str.upper().str.strip()
    out["Ticket"] = ticket
    out["TicketPrefix"] = (
        ticket
        .str.replace(r"[^A-Z]", "", regex=True)
        .replace("", "NUMERIC")
    )

    # Interaction feature
    out["SexClass"] = out["Sex"] + "_" + out["Pclass"]

    # CatBoost expects categorical columns to contain strings, not NaN values
    for column in CAT_COLUMNS:
        out[column] = out[column].fillna("Unknown").astype(str)

    return out

X = build_features(train)
X_test = build_features(test)

X.head()

## 6. Improved model — CatBoost ensemble

CatBoost is well suited to this dataset because it can work directly with categorical variables.

Two models with different tree depths are trained in each fold.  
Their predicted probabilities are averaged to reduce dependence on a single model configuration.

In [ ]:
def make_catboost(depth: int, seed: int = RANDOM_STATE):
    return CatBoostClassifier(
        iterations=800,
        depth=depth,
        learning_rate=0.03,
        l2_leaf_reg=5,
        loss_function="Logloss",
        cat_features=CAT_COLUMNS,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False
    )

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    fold_probabilities = []

    for depth in [4, 6]:
        model = make_catboost(depth)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])

        fold_probabilities.append(
            model.predict_proba(X.iloc[valid_idx])[:, 1]
        )

    mean_probability = np.mean(fold_probabilities, axis=0)
    fold_prediction = (mean_probability >= 0.5).astype(int)

    score = accuracy_score(y.iloc[valid_idx], fold_prediction)
    fold_scores.append(score)

    print(f"Fold {fold}: {score:.4f}")

print("\nCatBoost mean CV accuracy:", round(np.mean(fold_scores), 4))

## 7. Train on all data and create the Kaggle submission

After validation, the two CatBoost models are trained on all 891 labeled passengers.  
The final prediction is obtained by averaging both probability outputs.

The resulting file contains exactly the two columns expected by Kaggle:
`PassengerId` and `Survived`.

In [ ]:
test_probabilities = []

for depth in [4, 6]:
    final_model = make_catboost(depth)
    final_model.fit(X, y)
    test_probabilities.append(
        final_model.predict_proba(X_test)[:, 1]
    )

final_probability = np.mean(test_probabilities, axis=0)
final_prediction = (final_probability >= 0.5).astype(int)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": final_prediction
})

assert submission.shape == (418, 2)
assert submission.isna().sum().sum() == 0
assert set(submission["Survived"].unique()).issubset({0, 1})

OUTPUT_PATH = Path("/kaggle/working/submission_final.csv")
submission.to_csv(OUTPUT_PATH, index=False)

print("Submission created:", OUTPUT_PATH)
print("Shape:", submission.shape)
print("Predicted survival rate:", round(submission["Survived"].mean(), 3))

submission.head()

## 8. Result and conclusion

This project started with a simple Logistic Regression baseline and was improved through:
- cleaner validation,
- categorical feature handling,
- family-related features,
- passenger title extraction,
- cabin and ticket information,
- an ensemble of CatBoost models.

**Best Kaggle public leaderboard score achieved during experimentation: 0.80622.**

### What this project demonstrates
- Python data manipulation with pandas
- Missing-value handling
- Feature engineering
- Classification models
- Stratified cross-validation
- Model comparison and iterative improvement
- Creation of a valid Kaggle submission

The next natural step would be to test additional features or model calibration only if they improve cross-validation consistently, rather than optimizing blindly for a single leaderboard score.